# Self-Driving Car -- Evolutionary Explorer v5

**Architecture:** GA + Neural Network | Novelty Search | NEAT-Inspired Speciation | Catastrophic Reset | Adaptive Mutation

---

## Changelog: v4.1 -> v5

| Module | Description |
|---|---|
| Novelty Search | Behavioral descriptor vectors + global archive; exploration pressure via k-NN distance scoring |
| Speciation (NEAT-inspired) | Genome distance clustering; reproduction isolated within species |
| Catastrophic Reset | Long-term stagnation detection; population shock with elite preservation |
| Adaptive Mutation Rate | Dynamic rate scaling in response to fitness progress or plateau |
| Random Genome Injection | 10% fresh random genomes injected each generation |
| Environment Randomization | Spawn position and angle jitter to prevent single-track overfitting |
| Extended Logging | Species count, novelty mean, mutation rate, stagnation counter added to history |

All v4.1 charts are preserved. New evolutionary dynamics charts added in Cell 12.

**Usage:** Runtime -> Run all

In [ ]:
# =============================================================================
# CELL 1 -- Dependencies
# =============================================================================
!pip install pillow numpy matplotlib -q
!apt-get install -y ffmpeg -q

import warnings
warnings.filterwarnings('ignore', message='Glyph.*missing from font')

import numpy as np
import math
import random
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FFMpegWriter
from IPython.display import display, Video

print('[OK] All dependencies loaded.')

In [ ]:
# =============================================================================
# CELL 2 -- Hyperparameter Configuration
# =============================================================================

# -- Environment
W, H          = 500, 400
CAR_W, CAR_H  = 10, 16
CAR_SPEED     = 2.0
TURN_SPEED    = 2.2

# -- Sensor
N_SENSORS  = 7
SENSOR_LEN = 30

# -- Neural network
NN_HIDDEN = 8

# -- Simulation
MAX_FRAMES    = 800
N_GENERATIONS = 100
SAMPLE_EVERY  = 5
GIF_FPS       = 20
RENDER_GENS   = [1] + list(range(10, N_GENERATIONS + 1, 10))

# -- Population
POP_START = 2
POP_MAX   = 100

# -- Base mutation (adaptive; will change during training)
MUTATION_RATE    = 0.12
MUTATION_STR_MIN = 0.05
MUTATION_STR_MAX = 0.50
CHAOS_PROB       = 0.10
CHAOS_STR_MIN    = 0.50
CHAOS_STR_MAX    = 2.00
PARENT_POOL_SIZE = 10
ELITE_COUNT      = 1

# -- Adaptive mutation bounds
MUTATION_RATE_MIN   = 0.05
MUTATION_RATE_MAX   = 0.40
MUTATION_RATE_DECAY = 0.95   # applied on fitness improvement
MUTATION_RATE_BOOST = 1.20   # applied after STAGNATION_SOFT consecutive no-improvement gens
STAGNATION_SOFT     = 10

# -- Novelty search
NOVELTY_K           = 10
NOVELTY_WEIGHT      = 0.15
NOVELTY_THRESHOLD   = 3.0
NOVELTY_ARCHIVE_MAX = 500

# -- Speciation
SPECIES_THRESHOLD = 1.5
MAX_SPECIES       = 10
MIN_SPECIES_SIZE  = 2

# -- Catastrophic reset
STAGNATION_LIMIT     = 25
RESET_ELITE_COUNT    = 5
RESET_MUTATION_BOOST = 1.5

# -- Random genome injection
RANDOM_INJECT_FRAC = 0.10

# -- Spawn randomization
SPAWN_JITTER_X     = 20    # +/- px
SPAWN_JITTER_Y     = 20    # +/- px
SPAWN_JITTER_ANGLE = 25    # +/- degrees

print(f'[OK] Configuration loaded. Render generations: {RENDER_GENS}')

In [ ]:
# =============================================================================
# CELL 3 -- Track Generation
# =============================================================================
random.seed(42)
np.random.seed(42)

def generate_track(w=W, h=H):
    img  = Image.new('RGB', (w, h), (30, 30, 30))
    draw = ImageDraw.Draw(img)
    cx, cy = w // 2, h // 2

    def get_ring_points(n=300, base_rx=210, base_ry=160, bumps=None):
        pts = []
        for i in range(n):
            t = 2 * math.pi * i / n
            rx, ry = base_rx, base_ry
            if bumps:
                for angle, strength, width in bumps:
                    diff = math.sin((t - angle) * width)
                    rx += strength * diff
                    ry += strength * diff * 0.6
            pts.append((cx + math.cos(t) * rx, cy + math.sin(t) * ry))
        return pts

    bumps = [
        (0.0,         30, 3),
        (math.pi/2,   25, 4),
        (math.pi,     35, 3),
        (3*math.pi/2, 20, 5),
        (math.pi/4,   15, 6),
        (5*math.pi/4, 20, 4),
    ]

    outer_pts = get_ring_points(300, base_rx=215, base_ry=165, bumps=bumps)
    inner_pts = get_ring_points(300, base_rx=145, base_ry=100, bumps=bumps)

    draw.polygon(outer_pts, fill=(200, 200, 200))
    draw.polygon(inner_pts, fill=(30, 30, 30))
    draw.line([(250, 310), (250, 370)], fill=(255, 50, 50), width=4)

    return img

TRACK_IMG  = generate_track()
TRACK_ARR  = np.array(TRACK_IMG)
TRACK_MASK = TRACK_ARR[:, :, 0] > 150

# Auto-scan for a valid spawn position
START_ANGLE  = 180.0
START_X, START_Y = 250, 360
for try_y in range(355, 300, -5):
    valid_xs = [x for x in range(150, 380) if TRACK_MASK[try_y, x]]
    if len(valid_xs) >= 10:
        START_X = valid_xs[len(valid_xs) // 2]
        START_Y = try_y
        break

spawn_safe = all([
    TRACK_MASK[START_Y, START_X],
    TRACK_MASK[START_Y, START_X - 10],
    TRACK_MASK[START_Y, START_X + 10]
])
print(f'[OK] Spawn point: ({START_X}, {START_Y}) | On-track check: {spawn_safe}')

plt.figure(figsize=(8, 6))
plt.imshow(TRACK_IMG)
plt.plot(START_X, START_Y, 'go', markersize=10, label='Spawn')
plt.title('Track Layout  |  Red = Start Line  |  Green = Spawn Point')
plt.axis('off')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 4 -- Neural Network
# =============================================================================
# Architecture : input (N_SENSORS) -> hidden (NN_HIDDEN) -> output (2)
# Activation   : tanh on both layers
# Output[0]    : steering delta
# Output[1]    : throttle
# Genome       : flat weight vector, length = N_SENSORS*NN_HIDDEN + NN_HIDDEN
#                                            + NN_HIDDEN*2 + 2
# =============================================================================

class NeuralNet:
    def __init__(self, weights=None):
        self.size = N_SENSORS * NN_HIDDEN + NN_HIDDEN + NN_HIDDEN * 2 + 2
        if weights is None:
            weights = np.random.randn(self.size) * 0.5
        self._unpack(weights)

    def _unpack(self, w):
        i = 0
        self.w1 = w[i:i + N_SENSORS * NN_HIDDEN].reshape(N_SENSORS, NN_HIDDEN)
        i += N_SENSORS * NN_HIDDEN
        self.b1 = w[i:i + NN_HIDDEN]
        i += NN_HIDDEN
        self.w2 = w[i:i + NN_HIDDEN * 2].reshape(NN_HIDDEN, 2)
        i += NN_HIDDEN * 2
        self.b2 = w[i:i + 2]

    def get_weights(self):
        return np.concatenate([self.w1.flatten(), self.b1, self.w2.flatten(), self.b2])

    def forward(self, x):
        h = np.tanh(x @ self.w1 + self.b1)
        return np.tanh(h @ self.w2 + self.b2)


print('[OK] NeuralNet initialized.')
print(f'     Genome size: {NeuralNet().size} parameters')

In [ ]:
# =============================================================================
# CELL 5 -- Car Agent
# =============================================================================
# v5 changes vs v4.1:
#   - Spawn jitter expanded to SPAWN_JITTER_X/Y/ANGLE (environment randomization)
#   - Behavioral descriptor accumulators added: _steer_acc, _collisions, _final_x/y
#   - get_behavior_vector(): returns normalized 5-dim descriptor for novelty search
# =============================================================================

class Car:
    def __init__(self, brain):
        self.brain = brain
        self.reset()

    def reset(self, jitter=True):
        ox = random.uniform(-SPAWN_JITTER_X, SPAWN_JITTER_X) if jitter else 0
        oy = random.uniform(-SPAWN_JITTER_Y, SPAWN_JITTER_Y) if jitter else 0
        oa = random.uniform(-SPAWN_JITTER_ANGLE, SPAWN_JITTER_ANGLE) if jitter else 0

        self.x       = float(START_X) + ox
        self.y       = float(START_Y) + oy
        self.angle   = float(START_ANGLE) + oa
        self.alive   = True
        self.fitness = 0.0
        self.sensors = [1.0] * N_SENSORS
        self.visited = set()

        # Behavioral descriptor accumulators
        self._steer_acc  = []
        self._collisions = 0
        self._final_x    = float(START_X)
        self._final_y    = float(START_Y)

    # -------------------------------------------------------------------------

    def _on_track(self, x, y):
        xi, yi = int(x), int(y)
        if xi < 0 or yi < 0 or xi >= W or yi >= H:
            return False
        return bool(TRACK_MASK[yi, xi])

    def _cast_ray(self, angle_deg):
        rad = math.radians(angle_deg)
        for d in range(1, SENSOR_LEN + 1):
            tx = self.x + math.cos(rad) * d
            ty = self.y + math.sin(rad) * d
            if not self._on_track(tx, ty):
                return d / SENSOR_LEN
        return 1.0

    def _corners(self, cx, cy):
        rad = math.radians(self.angle)
        hw, hh = CAR_W / 2, CAR_H / 2
        pts = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
        return [
            (lx * math.cos(rad) - ly * math.sin(rad) + cx,
             lx * math.sin(rad) + ly * math.cos(rad) + cy)
            for lx, ly in pts
        ]

    # -------------------------------------------------------------------------

    def update(self):
        if not self.alive:
            return

        offsets = [-90, -60, -30, 0, 30, 60, 90]
        self.sensors = [self._cast_ray(self.angle + o) for o in offsets]
        inp   = np.array(self.sensors, dtype=np.float32)
        out   = self.brain.forward(inp)
        steer = float(out[0])
        gas   = (float(out[1]) + 1) / 2

        self.angle += steer * TURN_SPEED
        speed = gas * CAR_SPEED + 0.5
        nx = self.x + math.cos(math.radians(self.angle)) * speed
        ny = self.y + math.sin(math.radians(self.angle)) * speed

        self._steer_acc.append(abs(steer))

        if any(not self._on_track(cx, cy) for cx, cy in self._corners(nx, ny)):
            self._collisions += 1
            self.alive = False
            self._final_x, self._final_y = self.x, self.y
            return

        self.x, self.y = nx, ny
        self._final_x, self._final_y = self.x, self.y
        self.visited.add((int(self.x / 10), int(self.y / 10)))
        self.fitness = len(self.visited)

    def get_behavior_vector(self):
        """
        Returns a normalized 5-dimensional behavioral descriptor:
          [final_x, final_y, distance_proxy, mean_steer_magnitude, collision_count]
        All components normalized to approximately [0, 1].
        """
        dist       = float(len(self.visited))
        steer_mean = float(np.mean(self._steer_acc)) if self._steer_acc else 0.0
        return np.array([
            self._final_x / W,
            self._final_y / H,
            dist / 200.0,
            steer_mean,
            float(self._collisions) / 10.0
        ], dtype=np.float32)


print('[OK] Car agent initialized.')

In [ ]:
# =============================================================================
# CELL 6 -- Novelty Search Engine
# =============================================================================
# Novelty score  = mean Euclidean distance to k nearest neighbors
# Neighbor pool  = current population + global archive
# Archive policy = add if novelty_score >= NOVELTY_THRESHOLD
#                  evict oldest entries when size > NOVELTY_ARCHIVE_MAX (FIFO)
# =============================================================================

novelty_archive = []


def compute_novelty_scores(behavior_vectors, k=NOVELTY_K):
    """
    Compute per-individual novelty scores.

    Parameters
    ----------
    behavior_vectors : list of np.ndarray, shape (5,)
    k                : number of nearest neighbors

    Returns
    -------
    np.ndarray, shape (N,)
    """
    global novelty_archive

    pool = list(behavior_vectors)
    if novelty_archive:
        pool = pool + novelty_archive

    pool_arr = np.array(pool, dtype=np.float32)
    bv_arr   = np.array(behavior_vectors, dtype=np.float32)

    scores = []
    for bv in bv_arr:
        dists        = np.linalg.norm(pool_arr - bv, axis=1)
        dists_sorted = np.sort(dists)
        # Skip self (distance near zero)
        knn = dists_sorted[1:k + 1] if dists_sorted[0] < 1e-8 else dists_sorted[:k]
        scores.append(float(np.mean(knn)) if len(knn) > 0 else 0.0)

    return np.array(scores, dtype=np.float32)


def update_novelty_archive(behavior_vectors, novelty_scores):
    """Add high-novelty behavior vectors to the global archive."""
    global novelty_archive
    for bv, ns in zip(behavior_vectors, novelty_scores):
        if ns >= NOVELTY_THRESHOLD:
            novelty_archive.append(list(bv))
    if len(novelty_archive) > NOVELTY_ARCHIVE_MAX:
        novelty_archive = novelty_archive[-NOVELTY_ARCHIVE_MAX:]


print('[OK] Novelty Search engine ready.')
print(f'     k={NOVELTY_K} | weight={NOVELTY_WEIGHT} | archive_threshold={NOVELTY_THRESHOLD}')

In [ ]:
# =============================================================================
# CELL 7 -- NEAT-Inspired Speciation
# =============================================================================
# Genome distance  : mean(|w_a - w_b|) over all weight parameters
# Assignment rule  : individual joins nearest species if dist < SPECIES_THRESHOLD;
#                    otherwise a new species is created (up to MAX_SPECIES cap)
# Reproduction     : crossover and mutation are strictly intra-species
# Slot allocation  : proportional to species mean fitness
# =============================================================================

class Species:
    def __init__(self, species_id, representative):
        self.id             = species_id
        self.representative = representative.copy()
        self.members        = []   # list of (genome_vector, fitness) tuples
        self.age            = 0

    def mean_fitness(self):
        if not self.members:
            return 0.0
        return float(np.mean([m[1] for m in self.members]))

    def update_representative(self):
        """Set representative to the highest-fitness member."""
        if self.members:
            best = max(self.members, key=lambda m: m[1])
            self.representative = best[0].copy()


# Module-level species pool
_species_pool    = []
_next_species_id = 0


def genome_distance(w1, w2):
    """Mean absolute difference across all genome parameters."""
    return float(np.mean(np.abs(w1 - w2)))


def assign_to_species(genomes_with_fitness):
    """
    Assign each genome to its nearest species or create a new one.

    Parameters
    ----------
    genomes_with_fitness : list of (genome_vector, fitness) tuples

    Returns
    -------
    list of active Species objects with members populated
    """
    global _species_pool, _next_species_id

    for sp in _species_pool:
        sp.members = []
        sp.age    += 1

    for genome, fitness in genomes_with_fitness:
        best_sp   = None
        best_dist = float('inf')
        for sp in _species_pool:
            d = genome_distance(genome, sp.representative)
            if d < best_dist:
                best_dist = d
                best_sp   = sp

        if best_sp is not None and best_dist < SPECIES_THRESHOLD:
            best_sp.members.append((genome, fitness))
        else:
            if len(_species_pool) < MAX_SPECIES:
                new_sp = Species(_next_species_id, genome)
                new_sp.members.append((genome, fitness))
                _species_pool.append(new_sp)
                _next_species_id += 1
            else:
                # Pool at capacity -- assign to nearest
                if best_sp is not None:
                    best_sp.members.append((genome, fitness))

    # Prune underpopulated species
    _species_pool = [sp for sp in _species_pool if len(sp.members) >= MIN_SPECIES_SIZE]

    for sp in _species_pool:
        sp.update_representative()

    return _species_pool


def reproduce_from_species(species_list, target_pop, current_mut_rate,
                            elite_count=1, random_inject_frac=RANDOM_INJECT_FRAC):
    """
    Produce the next generation via species-isolated reproduction.

    Composition of next generation:
      1. Global elite  : top `elite_count` genomes copied without mutation
      2. Random inject : `random_inject_frac` of target_pop as fresh random genomes
      3. Species breed : remaining slots distributed proportionally to species
                         mean fitness; uniform crossover + adaptive mutation

    Returns
    -------
    (list[NeuralNet], int marriages)
    """
    global _last_mutation_log

    new_weights      = []
    marriages        = 0
    _mutation_deltas = []

    # -- Global elitism
    all_members = [(g, f) for sp in species_list for g, f in sp.members]
    all_members.sort(key=lambda m: m[1], reverse=True)
    for i in range(min(elite_count, len(all_members))):
        new_weights.append(all_members[i][0].copy())

    # -- Random genome injection
    nn_size  = NeuralNet().size
    n_random = max(1, int(target_pop * random_inject_frac))
    for _ in range(n_random):
        new_weights.append(np.random.randn(nn_size) * 0.5)

    remaining = target_pop - len(new_weights)
    if remaining <= 0:
        return [NeuralNet(w) for w in new_weights[:target_pop]], marriages

    # -- Proportional slot allocation
    total_mean  = sum(max(sp.mean_fitness(), 1e-6) for sp in species_list)
    allocations = []
    for sp in species_list:
        share = (max(sp.mean_fitness(), 1e-6) / total_mean) * remaining
        allocations.append(max(1, int(share)))

    delta = remaining - sum(allocations)
    if delta > 0:
        allocations[0] += delta

    # -- Intra-species crossover + mutation
    for sp, n_offspring in zip(species_list, allocations):
        pool = [m[0] for m in sp.members]
        if not pool:
            continue
        for _ in range(n_offspring):
            if len(new_weights) >= target_pop:
                break
            pA = random.choice(pool).copy()
            pB = random.choice(pool).copy()
            if len(pool) > 1:
                attempts = 0
                while np.array_equal(pA, pB) and attempts < 10:
                    pB = random.choice(pool).copy()
                    attempts += 1
            before = uniform_crossover(pA, pB)
            child  = mutate_adaptive(before, current_mut_rate)
            _mutation_deltas.extend((child - before).tolist())
            new_weights.append(child)
            marriages += 1

    # -- Fallback fill
    while len(new_weights) < target_pop:
        if all_members:
            pA = random.choice(all_members)[0].copy()
            pB = random.choice(all_members)[0].copy()
            before = uniform_crossover(pA, pB)
            child  = mutate_adaptive(before, current_mut_rate)
            new_weights.append(child)
            marriages += 1
        else:
            new_weights.append(np.random.randn(nn_size) * 0.5)

    _last_mutation_log.clear()
    _last_mutation_log.extend(_mutation_deltas)

    return [NeuralNet(w) for w in new_weights[:target_pop]], marriages


print('[OK] Speciation module ready.')
print(f'     threshold={SPECIES_THRESHOLD} | max_species={MAX_SPECIES} | min_size={MIN_SPECIES_SIZE}')

In [ ]:
# =============================================================================
# CELL 8 -- Genetic Operators | Adaptive Mutation | Catastrophic Reset
# =============================================================================

TOTAL_MARRIAGES       = 0
current_mutation_rate = MUTATION_RATE

# Logging buffer -- consumed by Cell 11 mutation histogram
_last_mutation_log = []


def uniform_crossover(w1, w2):
    """Element-wise random selection from either parent genome."""
    mask = np.random.rand(len(w1)) < 0.5
    return np.where(mask, w1, w2)


def mutate_adaptive(w, mut_rate):
    """
    Gaussian mutation with adaptive rate.
    CHAOS_PROB fraction of mutations use high-variance distribution
    to allow occasional large weight perturbations.
    """
    w    = w.copy()
    mask = np.random.rand(len(w)) < mut_rate
    n    = mask.sum()
    if n == 0:
        return w
    chaos_mask = np.random.rand(n) < CHAOS_PROB
    strengths  = np.where(
        chaos_mask,
        np.random.uniform(CHAOS_STR_MIN, CHAOS_STR_MAX, n),
        np.random.uniform(MUTATION_STR_MIN, MUTATION_STR_MAX, n)
    )
    w[mask] += np.random.randn(n) * strengths
    return w


def adapt_mutation_rate(stagnation_soft_counter, improved):
    """
    Update the global mutation rate.

    Rules:
      - Fitness improved    -> rate *= MUTATION_RATE_DECAY  (tighten search)
      - Soft stagnation hit -> rate *= MUTATION_RATE_BOOST  (widen search)
      - Rate clamped to [MUTATION_RATE_MIN, MUTATION_RATE_MAX]
    """
    global current_mutation_rate
    if improved:
        current_mutation_rate *= MUTATION_RATE_DECAY
    elif stagnation_soft_counter >= STAGNATION_SOFT:
        current_mutation_rate *= MUTATION_RATE_BOOST
    current_mutation_rate = float(
        np.clip(current_mutation_rate, MUTATION_RATE_MIN, MUTATION_RATE_MAX)
    )


def catastrophic_reset(cars, current_pop):
    """
    Hard population reset triggered by long-term stagnation.

    Procedure:
      1. Preserve top RESET_ELITE_COUNT genomes unchanged.
      2. Replace remaining population with random genomes.
      3. Boost mutation rate by RESET_MUTATION_BOOST.
      4. Clear species pool to force re-clustering.

    Returns
    -------
    list[NeuralNet] -- new population for next generation
    """
    global current_mutation_rate, _species_pool

    cars.sort(key=lambda c: c.fitness, reverse=True)
    elite_weights  = [c.brain.get_weights().copy() for c in cars[:RESET_ELITE_COUNT]]

    nn_size        = NeuralNet().size
    n_random       = current_pop - len(elite_weights)
    random_weights = [np.random.randn(nn_size) * 0.5 for _ in range(n_random)]

    current_mutation_rate = min(
        current_mutation_rate * RESET_MUTATION_BOOST,
        MUTATION_RATE_MAX
    )
    _species_pool.clear()

    print(f'     [RESET] Elite preserved: {len(elite_weights)} | '
          f'Random injected: {n_random} | '
          f'New mutation rate: {current_mutation_rate:.4f}')

    return [NeuralNet(w) for w in elite_weights + random_weights]


def evolve_v5(cars, generation, current_pop):
    """
    Full v5 evolution pipeline:
      1. Collect behavioral descriptors
      2. Compute novelty scores (k-NN in behavior space)
      3. Update novelty archive
      4. Compute final_score = raw_fitness + NOVELTY_WEIGHT * novelty_score
      5. Assign genomes to species
      6. Species-proportional reproduction

    Returns
    -------
    (new_brains, marriages, next_pop, n_species, novelty_mean)
    """
    global TOTAL_MARRIAGES

    behavior_vecs = [c.get_behavior_vector() for c in cars]
    raw_fitnesses = [c.fitness for c in cars]

    novelty_scores = compute_novelty_scores(behavior_vecs)
    update_novelty_archive(behavior_vecs, novelty_scores)

    final_scores = [
        rf + NOVELTY_WEIGHT * ns
        for rf, ns in zip(raw_fitnesses, novelty_scores)
    ]

    genomes      = [c.brain.get_weights().copy() for c in cars]
    gf_pairs     = list(zip(genomes, final_scores))
    species_list = assign_to_species(gf_pairs)

    next_pop   = min(current_pop * 2, POP_MAX)
    new_brains, marriages = reproduce_from_species(
        species_list, next_pop, current_mutation_rate, ELITE_COUNT
    )
    TOTAL_MARRIAGES += marriages

    return new_brains, marriages, next_pop, len(species_list), float(np.mean(novelty_scores))


print('[OK] Genetic operators, adaptive mutation, and catastrophic reset ready.')

In [ ]:
# =============================================================================
# CELL 9 -- Frame Renderer
# =============================================================================
# v5 HUD additions: species count, mean novelty score, stagnation counter
# =============================================================================

def _draw_car(draw, car, color):
    rad = math.radians(car.angle)
    hw, hh = CAR_W / 2, CAR_H / 2
    pts = [
        (int((lx * math.cos(rad) - ly * math.sin(rad)) + car.x),
         int((lx * math.sin(rad) + ly * math.cos(rad)) + car.y))
        for lx, ly in [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
    ]
    draw.polygon(pts, fill=color)


def _draw_sensors(draw, car):
    for i, off in enumerate([-90, -60, -30, 0, 30, 60, 90]):
        rad  = math.radians(car.angle + off)
        dist = car.sensors[i] * SENSOR_LEN
        draw.line(
            [(int(car.x), int(car.y)),
             (int(car.x + math.cos(rad) * dist),
              int(car.y + math.sin(rad) * dist))],
            fill=(0, 255, 100), width=1
        )


def render_frame(cars, generation, marriages_this_gen, total_pop,
                 n_species=0, novelty_mean=0.0, stagnation=0):
    frame_img = TRACK_IMG.copy()
    draw      = ImageDraw.Draw(frame_img)

    alive = [c for c in cars if c.alive]
    dead  = [c for c in cars if not c.alive]

    if alive:
        dominant = max(alive, key=lambda c:
            math.sqrt((c.x - START_X)**2 + (c.y - START_Y)**2))
    elif dead:
        dominant = max(dead, key=lambda c: c.fitness)
    else:
        dominant = None

    for car in dead:
        if car is dominant:
            continue
        _draw_car(draw, car, (160, 30, 30))

    for car in alive:
        if car is dominant:
            continue
        _draw_car(draw, car, (220, 60, 60))

    if dominant:
        _draw_sensors(draw, dominant)
        _draw_car(draw, dominant, (50, 255, 80))

    top_fit  = int(max((c.fitness for c in cars), default=0))
    dom_dist = int(math.sqrt((dominant.x - START_X)**2 + (dominant.y - START_Y)**2)) if dominant else 0
    dom_w    = f'{dominant.brain.get_weights().mean():.3f}' if dominant else '-'

    draw.rectangle([5, 5, 250, 155], fill=(0, 0, 0))
    draw.text((10, 10),  f'GEN        : {generation}',               fill=(255, 255, 255))
    draw.text((10, 26),  f'ALIVE      : {len(alive)}/{total_pop}',   fill=(255, 255, 255))
    draw.text((10, 42),  f'TOP FITNESS: {top_fit}',                  fill=(100, 255, 150))
    draw.text((10, 58),  f'DOM DIST   : {dom_dist}px',               fill=(50,  255,  80))
    draw.text((10, 74),  f'DOM WEIGHT : {dom_w}',                    fill=(50,  255,  80))
    draw.text((10, 90),  f'MARRIAGES  : {marriages_this_gen}x',      fill=(255, 200,   0))
    draw.text((10, 106), f'TOTAL MARRY: {TOTAL_MARRIAGES}x',         fill=(255, 200,   0))
    draw.text((10, 122), f'SPECIES    : {n_species}',                fill=(100, 200, 255))
    draw.text((10, 138), f'NOV: {novelty_mean:.2f}  STAG: {stagnation}', fill=(200, 180, 255))

    return np.array(frame_img)


print('[OK] Renderer ready.')

In [ ]:
# =============================================================================
# CELL 10 -- Training Loop
# =============================================================================
# History tuple layout (12 fields, index-aligned with analysis cells):
#   [0]  generation
#   [1]  best_fitness
#   [2]  mean_fitness
#   [3]  std_fitness
#   [4]  best_distance_from_spawn
#   [5]  mutation_delta_log          (list[float])
#   [6]  alive_curve                 (list[int])
#   [7]  actual_population_size
#   [8]  n_species                   (v5)
#   [9]  novelty_mean                (v5)
#   [10] current_mutation_rate       (v5)
#   [11] stagnation_counter          (v5)
# =============================================================================
print('Starting training -- Evolutionary Explorer v5')
print('Active modules: Novelty Search | Speciation | Catastrophic Reset | Adaptive Mutation')
print('-' * 90)

# -- Reset all stateful module globals
TOTAL_MARRIAGES       = 0
current_mutation_rate = MUTATION_RATE
novelty_archive.clear()
_species_pool.clear()
_next_species_id = 0

current_pop   = POP_START
brains        = [NeuralNet() for _ in range(current_pop)]
cars          = [Car(b) for b in brains]
history       = []
gif_frames    = {}
marriages_log = {}

# -- Stagnation state
global_best_fitness   = -1.0
stagnation_counter    = 0    # hard counter -- triggers catastrophic reset
stagnation_soft       = 0    # soft counter -- triggers mutation rate boost
last_n_species        = 0
last_novelty_mean     = 0.0

for gen in range(1, N_GENERATIONS + 1):
    for car in cars:
        car.reset()

    recording          = gen in RENDER_GENS
    frames_this_gen    = []
    marriages_this_gen = marriages_log.get(gen - 1, 0)
    alive_curve        = []

    # -- Simulation
    for f in range(MAX_FRAMES):
        for car in cars:
            car.update()
        alive_curve.append(sum(c.alive for c in cars))
        if recording and f % SAMPLE_EVERY == 0:
            frames_this_gen.append(
                render_frame(cars, gen, marriages_this_gen, current_pop,
                             last_n_species, last_novelty_mean, stagnation_counter)
            )
        if all(not c.alive for c in cars):
            break

    # -- Generation statistics
    all_fits  = [c.fitness for c in cars]
    best_gen  = max(all_fits)
    mean_fit  = float(np.mean(all_fits))
    std_fit   = float(np.std(all_fits))
    best_car  = max(cars, key=lambda c: c.fitness)
    best_dist = float(math.sqrt((best_car.x - START_X)**2 + (best_car.y - START_Y)**2))

    # -- Stagnation tracking
    improved = best_gen > global_best_fitness
    if improved:
        global_best_fitness = best_gen
        stagnation_counter  = 0
        stagnation_soft     = 0
    else:
        stagnation_counter += 1
        stagnation_soft    += 1

    # -- Adaptive mutation
    adapt_mutation_rate(stagnation_soft, improved)

    # -- Catastrophic reset
    reset_triggered = False
    if stagnation_counter >= STAGNATION_LIMIT:
        print(f'  Gen {gen:3d} -- CATASTROPHIC RESET triggered (stagnation={stagnation_counter})')
        cars_after_reset = catastrophic_reset(cars, current_pop)
        stagnation_counter = 0
        stagnation_soft    = 0
        reset_triggered    = True

    # -- Append history record
    history.append((
        gen,
        best_gen,
        mean_fit,
        std_fit,
        best_dist,
        _last_mutation_log.copy(),
        alive_curve,
        current_pop,
        last_n_species,
        last_novelty_mean,
        current_mutation_rate,
        stagnation_counter
    ))

    if recording:
        gif_frames[gen] = frames_this_gen

    # -- Evolve
    if reset_triggered:
        new_brains        = cars_after_reset
        marriages         = 0
        next_pop          = current_pop
        last_n_species    = 0
        last_novelty_mean = 0.0
    else:
        new_brains, marriages, next_pop, n_sp, nov_mean = evolve_v5(cars, gen, current_pop)
        last_n_species    = n_sp
        last_novelty_mean = nov_mean

    marriages_log[gen] = marriages
    current_pop        = next_pop
    cars               = [Car(b) for b in new_brains]

    reset_tag = '  [RESET]' if reset_triggered else ''
    print(
        f'Gen {gen:3d} | Pop: {current_pop:3d} | '
        f'Best: {int(best_gen):4d} | Mean: {mean_fit:6.1f} | Std: {std_fit:5.1f} | '
        f'Spc: {last_n_species:2d} | Nov: {last_novelty_mean:.3f} | '
        f'MutR: {current_mutation_rate:.4f} | Stag: {stagnation_counter:2d}'
        + reset_tag
    )

print('-' * 90)
print('Training complete.')
print(f'  Total crossover events : {TOTAL_MARRIAGES}')
print(f'  Novelty archive size   : {len(novelty_archive)}')
print(f'  Final species count    : {last_n_species}')

In [ ]:
# =============================================================================
# CELL 11 -- Analysis: Core Training Metrics (v4.1 layout preserved)
# =============================================================================
# Grid: 2x3
#   [0,0] Best Fitness per Generation
#   [0,1] Mean Fitness per Generation
#   [0,2] Fitness Standard Deviation
#   [1,0] Population Growth
#   [1,1] Best Distance from Spawn
#   [1,2] Mutation Strength Distribution (final generation)
# Plus: Alive Agents Over Time (final recorded generation)
# =============================================================================

gens      = [h[0]  for h in history]
fits      = [h[1]  for h in history]
means     = [h[2]  for h in history]
stds      = [h[3]  for h in history]
dists     = [h[4]  for h in history]
mut_log   = history[-1][5] if history else []
alive_log = history[-1][6] if history else []
pops      = [h[7]  for h in history]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0d0d1a')


def style_ax(ax, title, xlabel, ylabel):
    ax.set_facecolor('#1a1a2e')
    ax.set_title(title, color='white', fontsize=10, pad=6, fontweight='bold')
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    ax.tick_params(colors='#888888', labelsize=7)
    ax.grid(True, alpha=0.2, color='#444444')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')


ax = axes[0][0]
ax.plot(gens, fits, color='#00e5ff', linewidth=2)
ax.fill_between(gens, fits, alpha=0.15, color='#00e5ff')
style_ax(ax, 'Best Fitness per Generation', 'Generation', 'Best Fitness')

ax = axes[0][1]
ax.plot(gens, means, color='#a8e6cf', linewidth=2)
ax.fill_between(gens, means, alpha=0.15, color='#a8e6cf')
style_ax(ax, 'Mean Fitness per Generation', 'Generation', 'Mean Fitness')

ax = axes[0][2]
ax.plot(gens, stds, color='#ffcc02', linewidth=2)
ax.fill_between(gens, stds, alpha=0.15, color='#ffcc02')
style_ax(ax, 'Fitness Standard Deviation', 'Generation', 'Std Dev')

ax = axes[1][0]
ax.plot(gens, pops, color='#ff6b6b', linewidth=2)
ax.fill_between(gens, pops, alpha=0.15, color='#ff6b6b')
style_ax(ax, 'Population Growth', 'Generation', 'Population Size')

ax = axes[1][1]
ax.plot(gens, dists, color='#c3a6ff', linewidth=2)
ax.fill_between(gens, dists, alpha=0.15, color='#c3a6ff')
style_ax(ax, 'Best Distance from Spawn', 'Generation', 'Distance (px)')

ax = axes[1][2]
deltas = [abs(v) for v in mut_log if v != 0]
if deltas:
    ax.hist(deltas, bins=20, color='#ff9a3c', edgecolor='#222222', alpha=0.85)
else:
    ax.text(0.5, 0.5, 'No mutation data available', ha='center', va='center',
            transform=ax.transAxes, color='#888888', fontsize=9)
style_ax(ax, 'Mutation Strength Distribution (Final Gen)', '|Delta Weight|', 'Frequency')

plt.suptitle('Self-Driving Car v5 -- Core Training Metrics',
             color='white', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# -- Alive agents over time (final recorded generation)
if alive_log:
    fig2, ax2 = plt.subplots(figsize=(10, 3))
    fig2.patch.set_facecolor('#0d0d1a')
    ax2.set_facecolor('#1a1a2e')
    ax2.plot(alive_log, color='#50fa7b', linewidth=1.5)
    ax2.fill_between(range(len(alive_log)), alive_log, alpha=0.2, color='#50fa7b')
    ax2.set_title('Alive Agents Over Time -- Final Recorded Generation',
                  color='white', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Frame', color='#aaaaaa', fontsize=8)
    ax2.set_ylabel('Alive Agents', color='#aaaaaa', fontsize=8)
    ax2.tick_params(colors='#888888', labelsize=7)
    ax2.grid(True, alpha=0.2, color='#444444')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#333333')
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# CELL 12 -- Analysis: Evolutionary Dynamics (v5)
# =============================================================================
# Grid: 2x2
#   [0,0] Species Count per Generation
#   [0,1] Mean Novelty Score per Generation
#   [1,0] Adaptive Mutation Rate Trajectory
#   [1,1] Stagnation Counter Trajectory
# Plus: Run summary report
# =============================================================================

n_species_hist  = [h[8]  for h in history]
novelty_hist    = [h[9]  for h in history]
mut_rate_hist   = [h[10] for h in history]
stagnation_hist = [h[11] for h in history]

# Detect catastrophic resets: stagnation drops to 0 after reaching STAGNATION_LIMIT
total_resets = sum(
    1 for idx in range(1, len(stagnation_hist))
    if stagnation_hist[idx] == 0 and stagnation_hist[idx - 1] >= STAGNATION_LIMIT
)

fig3, axes3 = plt.subplots(2, 2, figsize=(12, 7))
fig3.patch.set_facecolor('#0d0d1a')

ax = axes3[0][0]
ax.plot(gens, n_species_hist, color='#00bcd4', linewidth=2, marker='o', markersize=3)
ax.fill_between(gens, n_species_hist, alpha=0.15, color='#00bcd4')
ax.axhline(y=MAX_SPECIES, color='#ff6b6b', linestyle='--', linewidth=1, alpha=0.6,
           label=f'Cap ({MAX_SPECIES})')
ax.legend(fontsize=7, labelcolor='white', facecolor='#1a1a2e', edgecolor='#333333')
style_ax(ax, 'Species Count per Generation', 'Generation', 'Species Count')

ax = axes3[0][1]
ax.plot(gens, novelty_hist, color='#ab47bc', linewidth=2)
ax.fill_between(gens, novelty_hist, alpha=0.15, color='#ab47bc')
ax.axhline(y=NOVELTY_THRESHOLD, color='#ffcc02', linestyle='--', linewidth=1, alpha=0.6,
           label=f'Archive threshold ({NOVELTY_THRESHOLD})')
ax.legend(fontsize=7, labelcolor='white', facecolor='#1a1a2e', edgecolor='#333333')
style_ax(ax, 'Mean Novelty Score per Generation', 'Generation', 'Mean Novelty')

ax = axes3[1][0]
ax.plot(gens, mut_rate_hist, color='#ff7043', linewidth=2)
ax.fill_between(gens, mut_rate_hist, alpha=0.15, color='#ff7043')
ax.axhline(y=MUTATION_RATE_MIN, color='#a8e6cf', linestyle=':', linewidth=1, alpha=0.6, label='Min')
ax.axhline(y=MUTATION_RATE_MAX, color='#ff6b6b', linestyle=':', linewidth=1, alpha=0.6, label='Max')
ax.legend(fontsize=7, labelcolor='white', facecolor='#1a1a2e', edgecolor='#333333')
style_ax(ax, 'Adaptive Mutation Rate', 'Generation', 'Mutation Rate')

ax = axes3[1][1]
ax.plot(gens, stagnation_hist, color='#ef5350', linewidth=2)
ax.fill_between(gens, stagnation_hist, alpha=0.15, color='#ef5350')
ax.axhline(y=STAGNATION_LIMIT, color='#ffcc02', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Reset threshold ({STAGNATION_LIMIT})')
ax.legend(fontsize=7, labelcolor='white', facecolor='#1a1a2e', edgecolor='#333333')
style_ax(ax, 'Stagnation Counter', 'Generation', 'Gens Without Improvement')

plt.suptitle('Self-Driving Car v5 -- Evolutionary Dynamics',
             color='white', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n=== Run Summary ===')
print(f'  Global best fitness    : {max(fits):.1f}')
print(f'  Max species observed   : {max(n_species_hist)}')
print(f'  Novelty archive size   : {len(novelty_archive)}')
print(f'  Peak mean novelty      : {max(novelty_hist):.4f}')
print(f'  Final mutation rate    : {mut_rate_hist[-1]:.4f}')
print(f'  Catastrophic resets    : {total_resets}')
print(f'  Total crossover events : {TOTAL_MARRIAGES}')

In [ ]:
# =============================================================================
# CELL 13 -- Export Simulation Videos (MP4)
# =============================================================================
os.makedirs('/content/videos', exist_ok=True)
video_paths = []

for gen, frames in sorted(gif_frames.items()):
    if not frames:
        continue
    path = f'/content/videos/gen_{gen:03d}.mp4'
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    plt.tight_layout(pad=0)
    im = ax.imshow(frames[0])

    def make_update(frms):
        def update(i):
            im.set_data(frms[i])
            return [im]
        return update

    ani = animation.FuncAnimation(
        fig, make_update(frames),
        frames=len(frames),
        interval=1000 / GIF_FPS,
        blit=True
    )
    writer = FFMpegWriter(fps=GIF_FPS)
    ani.save(path, writer=writer)
    plt.close()
    video_paths.append((gen, path))
    print(f'  Saved: {path}  ({len(frames)} frames)')

print(f'Export complete. {len(video_paths)} video(s) written.')

In [ ]:
# =============================================================================
# CELL 14 -- Playback
# =============================================================================
for gen, path in video_paths:
    pop_at_gen = history[gen - 1][7] if gen <= len(history) else POP_MAX
    spc_at_gen = history[gen - 1][8] if gen <= len(history) else 0
    print(f'--- Generation {gen:3d} | Population: {pop_at_gen} | Species: {spc_at_gen} ---')
    display(Video(path, embed=True, width=600))
    print()

In [ ]:
# =============================================================================
# CELL 15 -- Package and Download
# =============================================================================
from google.colab import files
import zipfile

zip_path = '/content/self_driving_v5_evolutionary_explorer.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for gen, path in video_paths:
        zf.write(path, os.path.basename(path))

files.download(zip_path)
print(f'Archive ready: {zip_path}')